# Demo 05 : Retries and fallbacks

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · backoff

We build on module 04's retry policy and record increasing delays around an injected transient stock-service failure. We disable jitter only to make the demonstration visible. Retrying a read does not authorise repeating a purchase.

In [2]:
import json
import operator
import sqlite3
import time
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Usage(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.calls += 1

    def on_llm_end(self, response, **kwargs):
        for batch in response.generations:
            for generation in batch:
                usage = generation.message.usage_metadata or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)

usage = Usage()
llm = make_llm(callbacks=[usage], max_retries=0, max_tokens=650)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as conn:
    conn.row_factory = sqlite3.Row
    work_order = dict(conn.execute("SELECT * FROM work_orders WHERE wo_id = ?", ("BW-7001",)).fetchone())
    part = dict(conn.execute("SELECT * FROM parts WHERE part_no = ?", ("CE-CF-02",)).fetchone())
print("work order:", work_order)
print("part:", part)

def config(label):
    return {"configurable": {"thread_id": f"07-{label}-{uuid4()}"}, "recursion_limit": 30}

work order: {'wo_id': 'BW-7001', 'fleet_no': 'CE12-2021-0117', 'opened': '2026-09-14', 'status': 'open', 'error_code': 'F-CE-110', 'summary': 'Traction system over-temperature during afternoon route', 'mechanic_id': 'M01'}
part: {'part_no': 'CE-CF-02', 'model': 'CE-12', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}


In [3]:
from langgraph.types import RetryPolicy

class Stock(TypedDict):
    part_no: str
    stock: int
    source: str
    answer: str

attempts = []
def remote_stock(state):
    attempts.append(time.perf_counter())
    print("attempt:", len(attempts))
    if len(attempts) < 3:
        raise ConnectionError("injected transient stock-service outage")
    return {"stock": part["stock"], "source": "simulated live service"}

def explain(state):
    return {"answer": llm.invoke(f"Write a short Veenhaven Bus stock report using only: {state}. Do not reserve stock.").content}

retry = StateGraph(Stock)
retry.add_node("lookup", remote_stock, retry_policy=RetryPolicy(
    max_attempts=3, initial_interval=0.1, backoff_factor=2, max_interval=0.3,
    jitter=False, retry_on=ConnectionError))
retry.add_node("explain", explain)
retry.add_edge(START, "lookup")
retry.add_edge("lookup", "explain")
retry.add_edge("explain", END)
initial = {"part_no": part["part_no"], "stock": 0, "source": "", "answer": ""}
out = retry.compile().invoke(initial)
print("gaps:", [round(b - a, 3) for a, b in zip(attempts, attempts[1:])])
print("answer:", out["answer"])
assert len(attempts) == 3 and out["stock"] == part["stock"]

attempt: 1
attempt: 2


attempt: 3


gaps: [0.102, 0.2]
answer: Veenhaven Bus stock report: Part CE-CF-02 has 6 in stock, based on simulated live service. No stock reserved.


## b · fallback-chain

We try a live lookup, a dated local snapshot and a manual-review result in that order. We disclose the source and never let a snapshot authorise a reservation. Module 04's model fallback handles a model outage; this chain handles missing business data.

In [4]:
from langchain_core.runnables import RunnableLambda

def live(request):
    print("source: live unavailable")
    raise ConnectionError("injected live outage")

def snapshot(request):
    if request["part_no"] != part["part_no"]:
        print("source: snapshot missing")
        raise LookupError("part absent from snapshot")
    print("source: snapshot")
    return {"part_no": part["part_no"], "stock": part["stock"],
            "source": "course snapshot, freshness unverified", "may_reserve": False}

def manual(request):
    print("source: manual review")
    return {"part_no": request["part_no"], "stock": None,
            "source": "manual stock check required", "may_reserve": False}

chain = RunnableLambda(live).with_fallbacks(
    [RunnableLambda(snapshot), RunnableLambda(manual)],
    exceptions_to_handle=(ConnectionError, LookupError))
known = chain.invoke({"part_no": part["part_no"]})
unknown = chain.invoke({"part_no": "UNKNOWN"})
print("known:", known)
print("unknown:", unknown)
report = llm.invoke(f"Explain these Veenhaven Bus stock results concisely. Preserve uncertainty and do not authorise a reservation: {known}; {unknown}").content
print("report:", report)
assert known["stock"] == part["stock"] and unknown["stock"] is None
assert not known["may_reserve"] and not unknown["may_reserve"]

source: live unavailable
source: snapshot
source: live unavailable
source: snapshot missing
source: manual review
known: {'part_no': 'CE-CF-02', 'stock': 6, 'source': 'course snapshot, freshness unverified', 'may_reserve': False}
unknown: {'part_no': 'UNKNOWN', 'stock': None, 'source': 'manual stock check required', 'may_reserve': False}


report: The course snapshot reports 6 units of part CE-CF-02, but its freshness is unverified, so current availability is uncertain. For the other item, stock is unknown and requires a manual check. Neither result authorizes a reservation.


In [5]:
print("metrics:", json.dumps({"model": MODEL, "calls": usage.calls,
      "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens,
      "seconds": round(time.perf_counter() - started, 1)}))

metrics: {"model": "openai/gpt-6-luna", "calls": 2, "input_tokens": 145, "output_tokens": 86, "seconds": 6.9}


## Things to try

Optional : we can change the transient-failure count or remove the local snapshot and observe the manual-review fallback.